In [ ]:
import numpy as np
from scipy.linalg import eig
from pathlib import Path
from copy import deepcopy as copy

from TEST.geometry import Slab
from TEST.material import Material

e3 = np.array([2.00E+01,                     1.00E-01,                               6.25E-07, 1.00E-11]) # MeV

In [ ]:
mat = "LWR_3G_A.json"
P1consistent = False # True # 
pwd = Path.cwd()
dataMG_ref = Material(uniName=mat, energygrid=e3, datapath=str(pwd.joinpath(mat)), P1consistent=P1consistent)
dataMG_ref.S0[2,0] = 0.0

G = dataMG_ref.nE
# avgE = 1/2*(e3[:-1]+e3[1:])*1.602176634E-13  # J
# v2 = np.sqrt(2*avgE/1.674927351e-27)
# dataMG_ref.inv_vel = 1/(v2*100)  # s/cm

In [110]:
H = 152 # cm
B = np.pi/H
print(f"Buckling with H={H} cm is {B**2:1.3e} cm^-2")

Buckling with H=152 cm is 4.272e-04 cm^-2


In [111]:
# operator definition
L_ref = np.zeros((G, G))
Linf_ref = np.zeros((G, G))
F_ref = np.zeros((G, G))
S_ref = np.zeros((G, G))

for g in range(G):
    L_ref[g, g] = dataMG_ref.Sigma_rem[g] + dataMG_ref.Diffcoef[g] * B**2
    Linf_ref[g, g] = dataMG_ref.Sigma_rem[g]
    for h in range(G):
        F_ref[g, h] = dataMG_ref.chi_tot[g]*dataMG_ref.nu_fiss[h]*dataMG_ref.Sigma_fiss[h]
        if h != g: # this term is included in the removal XS
            S_ref[g, h] = dataMG_ref.S0[g, h]

Direct problem:
* $(\hat L - \hat S) \vec{\phi} = \displaystyle\frac{1}{k} \hat F \vec{\phi} $

Adjoint problem:
* $(\hat L^T - \hat S^T) \vec{\phi^+} = \displaystyle\frac{1}{k^+} \hat F^T \vec{\phi^+} $

where $k=k^+$ and $\braket{\hat F^T \vec{\phi^T} | \vec{\phi}} = \braket{\vec{\phi^T} | \hat F\vec{\phi}}$.

In [112]:
evlD, evcD = eig(L_ref-S_ref, b=F_ref)
kD = 1/evlD[0].real
flxD = evcD[:, 0] if evcD[:, 0].max() > 0 else -evcD[:, 0]

evlA, evcA = eig(L_ref.T-S_ref.T, b=F_ref.T)
kA = 1/evlA[0].real
flxA = evcA[:, 0] if evcA[:, 0].max() > 0 else -evcA[:, 0]

# sanity check on eigenvalues
assert 1E5*(abs(kA-kD)) < 1E-3
# sanity check on eigenvectors
assert abs(np.dot(flxA, F_ref.dot(flxD))-np.dot(F_ref.T.dot(flxA), flxD)) < 1E-12

# impose criticality
k = kD
F_ref = F_ref/k
evlD, evcD = eig(L_ref-S_ref, b=F_ref)
kD_ref = 1/evlD[0].real
flxD_ref = evcD[:, 0] if evcD[:, 0].max() > 0 else -evcD[:, 0]
flxD_ref /= F_ref.dot(flxD_ref).sum() # flxD_ref[0]

evlA, evcA = eig(L_ref.T-S_ref.T, b=F_ref.T)
kA_re = 1/evlA[0].real
flxA_ref = evcA[:, 0] if evcA[:, 0].max() > 0 else -evcA[:, 0]
flxA_ref /= flxA_ref[0]

# k infinity
evlinf, flx_inf = eig(Linf_ref-S_ref, b=F_ref)

kinf_ref = 1/evlinf[0].real

# sanity check on eigenvalues
assert 1E5*(abs(kA-kD)) < 1E-3
# sanity check on eigenvectors
assert abs(np.dot(flxA_ref, F_ref.dot(flxD_ref))-np.dot(F_ref.T.dot(flxA_ref), flxD_ref)) < 1E-12
print(f"Effective mult. param. with {G}G = {k:.10f}")

Effective mult. param. with 3G = 1.0493236025


In [113]:
Lambda_eff_ref = np.dot(flxA_ref, dataMG_ref.inv_vel*(flxD_ref)) / np.dot(flxA_ref, F_ref.dot(flxD_ref))
print(f"Total importance amount: {np.dot(flxA_ref, dataMG_ref.inv_vel*(flxD_ref))}")
print(f"Infinite mult. param. with {G}G = {kinf_ref:.6f}")
print(f"Effective mult. param. with {G}G = {kD_ref:.6f}")
print(f"Effective lifetime with {G}G = {Lambda_eff_ref*1E6:.4f} micro")

Total importance amount: 2.127030345556516e-05
Infinite mult. param. with 3G = 1.013931
Effective mult. param. with 3G = 1.000000
Effective lifetime with 3G = 21.2703 micro


In [114]:
chi = dataMG_ref.chi_tot
nsf = dataMG_ref.nuSigma_fiss/k
sc = dataMG_ref.S0.T
D = dataMG_ref.Diffcoef
r = dataMG_ref.Sigma_rem
v = 1/dataMG_ref.inv_vel
# leakage probabilities
P1 = 1/(D[0]*B**2+r[0])
P2 = 1/(D[1]*B**2+r[1])
P3 = 1/(D[2]*B**2+r[2])

s = (1-P2*P3*sc[1,2]*sc[2,1])
rs1 = dataMG_ref.Sigma_abs[1] + sc[1,2]*(1-sc[2,1]*P3)
rs2 = dataMG_ref.Sigma_abs[2] + sc[2,1]*(1-sc[1,2]*P2)

# up-scattering corrected leakage probabilities
P1s = 1/(D[0]*B**2+r[0])/s
P2s = 1/(D[1]*B**2+r[1])/s
P3s = 1/(D[2]*B**2+r[2])/s

C1 = (P1*sc[0,1]+P1*sc[0,2]*P3*sc[2,1])/s
C2 = (P1*sc[0,2]+P1*sc[0,1]*P2*sc[1,2])/s
k1 = nsf[0]*P1
k2 = nsf[1]*P2*C1
k3 = nsf[2]*P3*C2
keff_3G = k1+k2+k3
print(f"keff analyt. = {keff_3G:.10f}")

keff analyt. = 1.0000000000


In [115]:
flxA_ref /= flxA_ref[0]
flxD_ref /= flxD_ref[0]

phi1d = 1
phi2d = sc[0,1]*P2+sc[2,1]*P2s*(sc[0,2]*P3+sc[0,1]*sc[1,2]*P2*P3)
phi3d = sc[0,2]*P3s+P2s*P3*sc[0,1]*sc[1,2]

phi1a = 1
phi2a = P2*nsf[1]+sc[1,2]*P2*(nsf[2]*P3s+sc[2,1]*nsf[1]*P2*P3s)
phi3a = nsf[2]*P3s+sc[2,1]*nsf[1]*P2*P3s

phiD = np.array([phi1d, phi2d, phi3d])
phiA = np.array([phi1a, phi2a, phi3a])

den_ana = np.dot(phiA, F_ref.dot(phiD))
den_num = np.dot(flxA_ref, F_ref.dot(flxD_ref))
print(f"Effective lifetime denominator: analyt. = {den_ana:.4e}, numerical = {den_num:.4e}")

num_ana = np.dot(phiA, dataMG_ref.inv_vel*(phiD))
num_num = np.dot(flxA_ref, dataMG_ref.inv_vel*(flxD_ref))
print(f"Effective lifetime numerator: analyt. = {num_ana:.4e}, numerical = {num_num:.4e}")


Effective lifetime denominator: analyt. = 4.7021e-02, numerical = 4.7021e-02
Effective lifetime numerator: analyt. = 1.0002e-06, numerical = 1.0002e-06


# Analytical results

In [116]:
lambda1 = 1/v[0]*P1
lambda2 = 1/(v[1])*P2 #*(D[1]*B**2+rs1))
lambda3 = 1/(v[2])*P3 #*(D[2]*B**2+rs2))

w2 = nsf[1]*P2s*(sc[0,1]*P1s+sc[0,2]*sc[2,1]*P1s*P3) + \
     + nsf[2]*P3s*(sc[0,1]*sc[1,2]*P1*P2s) + \
     + nsf[2]*sc[0,2]*P1s*(1/(D[2]*B**2+rs2)-1/(D[2]*B**2+r[2]))
w3 = sc[0,1]*nsf[1]*P1s*(1/(D[1]*B**2+rs1)-1/(D[1]*B**2+r[1])) + \
     nsf[1]*P2s*(sc[0,2]*sc[2,1]*P1s*P3) + \
     nsf[2]*P3s*(sc[0,2]*P1s+sc[0,1]*sc[1,2]*P1s*P2)

print(f"l1: {lambda1*1E6:.4f}, l2: {lambda2*1E6:.4f} w2: {w2:.3f} , l3: {lambda3*1E6:.4f} w3: {w3:.3f}")

# q2 = 1/s*(k2+k3-P1*P3*sc[0,2]*nsf[2])
# q3 = (k2+k3)-P1*sc[0,1]*P2*nsf[1]
leff_3G = lambda1+lambda2*w2+lambda3*w3 # P1/v[0]+P2/v[1]*q2+P3/v[2]/s*q3
print(f"leff num. = {num_num/den_num*1E6:.4f} micros")
print(f"leff semi-analyt. = {num_ana/den_ana*1E6:.4f} micros")
print(f"leff analyt. = {leff_3G*1E6:.4f} micros")
print(f"Effective lifetime with {G}G = {Lambda_eff_ref*1E6:.4f} micros")

l1: 0.0190, l2: 2.2972 w2: 0.912 , l3: 25.2050 w3: 0.760
leff num. = 21.2703 micros
leff semi-analyt. = 21.2703 micros
leff analyt. = 21.2703 micros
Effective lifetime with 3G = 21.2703 micros


# Analytical results collapsing from 3G to 2G

In [117]:
e2 = np.array([2.00E+01,                                                             6.25E-07, 1.00E-11]) # MeV
e1 = np.array([2.00E+01,                                                                       1.00E-11]) # MeV

In [118]:
data2G = copy(dataMG_ref)
data2G.nu_fiss /= k
data2G.nuSigma_fiss = data2G.nu_fiss*data2G.Sigma_fiss
data2G.collapse(e2, spectrum=phiD, fixdata=True,)

nG = data2G.nE
# --- operator definition
L2G = np.zeros((nG, nG))
F2G = np.zeros((nG, nG))
S2G = np.zeros((nG, nG))
Linf2G = np.zeros((nG, nG))
for g in range(nG):
    L2G[g, g] = data2G.Sigma_rem[g] + data2G.Diffcoef[g] * B**2
    Linf2G[g, g] = data2G.Sigma_abs[g]+data2G.S0[:, g].sum()-data2G.S0[g, g]
    for h in range(nG):
        F2G[g, h] = data2G.chi_tot[g]*data2G.nu_fiss[h]*data2G.Sigma_fiss[h]
        if h != g: # this term is included in the removal XS
            S2G[g, h] = data2G.S0[g, h]
# --- eigenvalue calculation
evlD, evcD = eig(L2G-S2G, b=F2G)
kD2G = 1/evlD[0].real
flxD2G = evcD[:, 0] if evcD[:, 0].max() > 0 else -evcD[:, 0]
flxD2G /= flxD2G[0]

evlA, evcA = eig(L2G.T-S2G.T, b=F2G.T)
kA2G = 1/evlA[0].real
flxA2G = evcA[:, 0] if evcA[:, 0].max() > 0 else -evcA[:, 0]
flxA2G /= flxA2G[0]

# sanity check on eigenvalues
assert 1E5*(abs(kA-kD)) < 1E-3
# sanity check on flux
# assert np.allclose(flx_few, flxD)
# sanity check on eigenvectors bi-orthogonality
assert abs(np.dot(flxA2G, F2G.dot(flxD2G))-np.dot(F2G.T.dot(flxA2G), flxD2G)) < 1E-12

Lambda_eff_2G = np.dot(flxA2G, data2G.inv_vel*(flxD2G)) / np.dot(flxA2G, F2G.dot(flxD2G))

k2G_ref = kD2G

# Ensure criticality for the 2nd group

In [119]:
data2G.nu_fiss /= k2G_ref
data2G.nuSigma_fiss = data2G.nu_fiss*data2G.Sigma_fiss
# data2G.collapse(e2, spectrum=phiD, datacheck=True,)

nG = data2G.nE
# --- operator definition
L2G = np.zeros((nG, nG))
F2G = np.zeros((nG, nG))
S2G = np.zeros((nG, nG))
Linf2G = np.zeros((nG, nG))
for g in range(nG):
    L2G[g, g] = data2G.Sigma_rem[g] + data2G.Diffcoef[g] * B**2
    Linf2G[g, g] = data2G.Sigma_abs[g]+data2G.S0[:, g].sum()-data2G.S0[g, g]
    for h in range(nG):
        F2G[g, h] = data2G.chi_tot[g]*data2G.nu_fiss[h]*data2G.Sigma_fiss[h]
        if h != g: # this term is included in the removal XS
            S2G[g, h] = data2G.S0[g, h]
# --- eigenvalue calculation
evlD, evcD = eig(L2G-S2G, b=F2G)
kD2G = 1/evlD[0].real
flxD2G = evcD[:, 0] if evcD[:, 0].max() > 0 else -evcD[:, 0]
flxD2G /= flxD2G[0]

evlA, evcA = eig(L2G.T-S2G.T, b=F2G.T)
kA2G = 1/evlA[0].real
flxA2G = evcA[:, 0] if evcA[:, 0].max() > 0 else -evcA[:, 0]
flxA2G /= flxA2G[0]

assert 1E5*(abs(kA-kD)) < 1E-3
assert abs(np.dot(flxA2G, F2G.dot(flxD2G))-np.dot(F2G.T.dot(flxA2G), flxD2G)) < 1E-12

Lambda_eff_2G = np.dot(flxA2G, data2G.inv_vel*(flxD2G)) / np.dot(flxA2G, F2G.dot(flxD2G))

In [120]:
rs1_2G = data2G.Sigma_abs[0] + data2G.S0[1,0]*(1-data2G.S0[0,1]/(data2G.Diffcoef[1]*B**2+data2G.Sigma_rem[1]))
rs2_2G = data2G.Sigma_abs[1] + data2G.S0[0,1]*(1-data2G.S0[1,0]/(data2G.Diffcoef[1]*B**2+data2G.Sigma_rem[1]))

In [121]:
P1s_2G = 1/(data2G.Diffcoef[0]*B**2+rs1_2G)
P1_2G = 1/(data2G.Diffcoef[0]*B**2+data2G.Sigma_rem[0])
P2s_2G = 1/(data2G.Diffcoef[1]*B**2+rs2_2G)
P2_2G = 1/(data2G.Diffcoef[1]*B**2+data2G.Sigma_rem[1])

k1_2G = data2G.nuSigma_fiss[0]*P1s_2G # /(data2G.Diffcoef[0]*B**2+data2G.Sigma_rem[0])
k2_2G = data2G.nuSigma_fiss[1]*data2G.S0[1,0]*P1s_2G*P2_2G
k2G = k1_2G+k2_2G

print(f"k 2G: {(kD2G):.5f}")

k 2G: 1.00000


In [122]:
phi1d_2G = 1
phi2d_2G = data2G.S0[1,0]/(data2G.Diffcoef[1]*B**2+data2G.Sigma_rem[1])
phiD_2G = np.array([phi1d_2G, phi2d_2G])

phi1a_2G = 1
phi2a_2G = (data2G.S0[0,1]+data2G.nuSigma_fiss[1]/k2G)/(data2G.Diffcoef[1]*B**2+data2G.Sigma_rem[1])
phiA_2G = np.array([phi1a_2G, phi2a_2G])

w2_2G = sc[0,1]*P1s
w3_2G = w2_2G*sc[1,2]*P2*(sc[2,1]+nsf[2]/k2G)*P3
lambda_3G_2G = lambda1+lambda2*w2_2G+lambda3*w3_2G # lambda1+lambda2*w2_2G+(data2G.inv_vel[1]*phi2d_2G*phi2a_2G)/(k2G/P1s_2G) # 
lambda_3G_2G_v2 = (data2G.inv_vel[0]+data2G.inv_vel[1]*phi2d_2G*phi2a_2G)/(k2G/P1s_2G)

lambda_3G_2G_data = 1/k2G*(data2G.inv_vel[0]*P1s_2G + data2G.inv_vel[1]*P2s_2G * data2G.S0[1, 0]*P1_2G*(data2G.S0[0,1] + data2G.nuSigma_fiss[1]/k2G)*P2_2G)

print(f"numerical 2G: {Lambda_eff_2G*1E6:.8f}")
print(f"analytical 2G with 3G data: {lambda_3G_2G*1E6:.8f}")
print(f"analytical 2G  with 3G data v2: {lambda_3G_2G_v2*1E6:.8f}")
print(f"analytical 2G with 2G data: {lambda_3G_2G_data*1E6:.8f}")

print(f"l1: {lambda1*1E6:.4f}, l2: {lambda2*1E6:.4f} w2 2G: {w2_2G:.3f} , l3: {lambda3*1E6:.4f} w3 2G: {w3_2G:.3f}")


numerical 2G: 21.35658577
analytical 2G with 3G data: 21.35928115
analytical 2G  with 3G data v2: 21.35658577
analytical 2G with 2G data: 21.21549710
l1: 0.0190, l2: 2.2972 w2 2G: 0.946 , l3: 25.2050 w3 2G: 0.760


In [123]:
data1G = copy(data2G)
# data1G.nu_fiss /= k2G
data1G.collapse(e1, spectrum=flxD2G, fixdata=True)

nG = data2G.nE

Lambda_eff_1G = data1G.inv_vel[0] / data1G.nuSigma_fiss[0]

In [124]:
data1G.nuSigma_fiss[0] / (data1G.Diffcoef[0]*B**2 + data1G.Sigma_abs[0])

1.0000000000000036

In [125]:
Lambda_eff_1G_v2 = 1/k2G*(data2G.inv_vel[0]*P1s_2G + data2G.inv_vel[1]*P1s_2G*data2G.S0[1,0]*P2_2G)

In [126]:
data2G.S0

array([[0.50516998, 0.00157406],
       [0.01569344, 1.22474   ]])

In [127]:
data2G.S0[1,0]*P2_2G

0.16351731576590445

In [128]:
1/k2G*(data2G.inv_vel[0]*P1s_2G + data2G.inv_vel[1]*P1s_2G*data2G.S0[1,0]*P2_2G)

1.7750436012768368e-05

In [129]:
print(f"analytical 2G: {Lambda_eff_2G*1E6:.8f}")
print(f"analytical 1G: {Lambda_eff_1G*1E6:.8f}")
print(f"analytical 1G v2: {Lambda_eff_1G_v2*1E6:.8f}")

analytical 2G: 21.35658577
analytical 1G: 17.75043601
analytical 1G v2: 17.75043601
